
# Model 3: Customer Segmentation

#Importing the required pacakages

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import joblib

#Importing the data

In [0]:
VOLUME_PATH = "/Volumes/workspace/default/raw_data/"

In [0]:
demographics = pd.read_csv(VOLUME_PATH + "customer_demographics.csv")
location = pd.read_csv(VOLUME_PATH + "customer_location.csv")
services = pd.read_csv(VOLUME_PATH + "customer_services.csv")
account_status = pd.read_csv(VOLUME_PATH + "customer_account_status.csv")
zipcode_population = pd.read_csv(VOLUME_PATH + "zipcode_population.csv")

#Cleaning

In [0]:
services.head(3)

,Customer ID,Offer,Phone Service,Avg Monthly Long Distance Charges,Multiple Lines,Internet Service,Internet Type,Avg Monthly GB Download,Online Security,Online Backup,Device Protection Plan,Premium Tech Support,Streaming TV,Streaming Movies,Streaming Music,Unlimited Data
0,0002-ORFBO,NaN,Yes,42.39,No,Yes,Cable,16.0,No,Yes,No,Yes,Yes,No,No,Yes
1,0003-MKNFE,NaN,Yes,10.69,Yes,Yes,Cable,10.0,No,No,No,No,No,Yes,Yes,No
2,0004-TLHLJ,Offer E,Yes,33.65,No,Yes,Fiber Optic,30.0,No,No,Yes,No,No,No,No,Yes


In [0]:
services.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 16 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Offer                              3166 non-null   object 
 2   Phone Service                      7043 non-null   object 
 3   Avg Monthly Long Distance Charges  6361 non-null   float64
 4   Multiple Lines                     6361 non-null   object 
 5   Internet Service                   7043 non-null   object 
 6   Internet Type                      5517 non-null   object 
 7   Avg Monthly GB Download            5517 non-null   float64
 8   Online Security                    5517 non-null   object 
 9   Online Backup                      5517 non-null   object 
 10  Device Protection Plan             5517 non-null   object 
 11  Premium Tech Support               5517 non-null   objec

In [0]:
services['Internet_Type_Clean'] = services['Internet Type'].fillna('No Internet Service')
services['Offer_Clean'] = services['Offer'].fillna('No Offer')


In [0]:
def missing_value_imp(x):
  if x.dtype == 'int' or x.dtype == 'float':
    x = x.fillna(x.mean())
  else:
    x = x.fillna(x.mode()[0])
  return x

In [0]:
services = services.apply(missing_value_imp)

In [0]:
services.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 18 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Offer                              7043 non-null   object 
 2   Phone Service                      7043 non-null   object 
 3   Avg Monthly Long Distance Charges  7043 non-null   float64
 4   Multiple Lines                     7043 non-null   object 
 5   Internet Service                   7043 non-null   object 
 6   Internet Type                      7043 non-null   object 
 7   Avg Monthly GB Download            7043 non-null   float64
 8   Online Security                    7043 non-null   object 
 9   Online Backup                      7043 non-null   object 
 10  Device Protection Plan             7043 non-null   object 
 11  Premium Tech Support               7043 non-null   objec

In [0]:
account_status.head(3)

,Customer ID,Number of Referrals,Tenure in Months,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,2,9,One Year,Yes,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,0,9,Month-to-Month,No,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,0,4,Month-to-Month,Yes,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices


In [0]:
account_status.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Customer ID                  7043 non-null   object 
 1   Number of Referrals          7043 non-null   int64  
 2   Tenure in Months             7043 non-null   int64  
 3   Contract                     7043 non-null   object 
 4   Paperless Billing            7043 non-null   object 
 5   Payment Method               7043 non-null   object 
 6   Monthly Charge               7043 non-null   float64
 7   Total Charges                7043 non-null   float64
 8   Total Refunds                7043 non-null   float64
 9   Total Extra Data Charges     7043 non-null   int64  
 10  Total Long Distance Charges  7043 non-null   float64
 11  Total Revenue                7043 non-null   float64
 12  Customer Status              7043 non-null   object 
 13  Churn Category    

In [0]:
account_status['Churn_Category_Clean'] = account_status['Churn Category'].fillna('Not Churned')
account_status['Churn_Reason_Clean'] = account_status['Churn Reason'].fillna('Not Churned')

In [0]:
account_status.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Customer ID                  7043 non-null   object 
 1   Number of Referrals          7043 non-null   int64  
 2   Tenure in Months             7043 non-null   int64  
 3   Contract                     7043 non-null   object 
 4   Paperless Billing            7043 non-null   object 
 5   Payment Method               7043 non-null   object 
 6   Monthly Charge               7043 non-null   float64
 7   Total Charges                7043 non-null   float64
 8   Total Refunds                7043 non-null   float64
 9   Total Extra Data Charges     7043 non-null   int64  
 10  Total Long Distance Charges  7043 non-null   float64
 11  Total Revenue                7043 non-null   float64
 12  Customer Status              7043 non-null   object 
 13  Churn Category    

In [0]:
account_status['Has_Discount'] = np.where(account_status['Monthly Charge'] < 0, 1, 0)
account_status['Monthly_Discount_Amount'] = np.where(
    account_status['Monthly Charge'] < 0, account_status['Monthly Charge'].abs(), 0)

#Merge into one master table

In [0]:
location.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Customer ID  7043 non-null   object 
 1   City         7043 non-null   object 
 2   Zip Code     7043 non-null   int64  
 3   Latitude     7043 non-null   float64
 4   Longitude    7043 non-null   float64
dtypes: float64(2), int64(1), object(2)
memory usage: 275.2+ KB


In [0]:
data = demographics.merge(location, on='Customer ID', how='left')
data = data.merge(services, on='Customer ID', how='left')
data = data.merge(account_status, on='Customer ID', how='left')
data = data.merge(zipcode_population, on='Zip Code', how='left')

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 45 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Offer                              7043 non-null   object 
 10  Phone Service                      7043 non-null   object 
 11  Avg Monthly Long Distance Charges  7043 non-null   float

In [0]:
data.head()

,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Offer,Phone Service,Avg Monthly Long Distance Charges,Multiple Lines,Internet Service,Internet Type,Avg Monthly GB Download,Online Security,Online Backup,Device Protection Plan,Premium Tech Support,Streaming TV,Streaming Movies,Streaming Music,Unlimited Data,Internet_Type_Clean,Offer_Clean,Number of Referrals,Tenure in Months,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason,Churn_Category_Clean,Churn_Reason_Clean,Has_Discount,Monthly_Discount_Amount,Population
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,Offer B,Yes,42.39,No,Yes,Cable,16.0,No,Yes,No,Yes,Yes,No,No,Yes,Cable,No Offer,2,9,One Year,Yes,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN,Not Churned,Not Churned,0,0.0,4498
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,Offer B,Yes,10.69,Yes,Yes,Cable,10.0,No,No,No,No,No,Yes,Yes,No,Cable,No Offer,0,9,Month-to-Month,No,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN,Not Churned,Not Churned,1,4.0,31297
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,Offer E,Yes,33.65,No,Yes,Fiber Optic,30.0,No,No,Yes,No,No,No,No,Yes,Fiber Optic,Offer E,0,4,Month-to-Month,Yes,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices,Competitor,Competitor had better devices,0,0.0,62069
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,Offer D,Yes,27.82,No,Yes,Fiber Optic,4.0,No,Yes,Yes,No,Yes,Yes,No,Yes,Fiber Optic,Offer D,1,13,Month-to-Month,Yes,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction,Dissatisfaction,Product dissatisfaction,0,0.0,46677
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,Offer B,Yes,7.38,No,Yes,Fiber Optic,11.0,No,No,No,Yes,Yes,No,No,Yes,Fiber Optic,No Offer,3,3,Month-to-Month,Yes,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability,Dissatisfaction,Network reliability,0,0.0,42853


In [0]:
data.columns

Index(['Customer ID', 'Gender', 'Age', 'Married', 'Number of Dependents',
       'City', 'Zip Code', 'Latitude', 'Longitude', 'Offer', 'Phone Service',
       'Avg Monthly Long Distance Charges', 'Multiple Lines',
       'Internet Service', 'Internet Type', 'Avg Monthly GB Download',
       'Online Security', 'Online Backup', 'Device Protection Plan',
       'Premium Tech Support', 'Streaming TV', 'Streaming Movies',
       'Streaming Music', 'Unlimited Data', 'Internet_Type_Clean',
       'Offer_Clean', 'Number of Referrals', 'Tenure in Months', 'Contract',
       'Paperless Billing', 'Payment Method', 'Monthly Charge',
       'Total Charges', 'Total Refunds', 'Total Extra Data Charges',
       'Total Long Distance Charges', 'Total Revenue', 'Customer Status',
       'Churn Category', 'Churn Reason', 'Churn_Category_Clean',
       'Churn_Reason_Clean', 'Has_Discount', 'Monthly_Discount_Amount',
       'Population'],
      dtype='object')

In [0]:
data.drop(columns=['Offer','Internet Type','Churn Category','Churn Reason'], inplace=True)

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 41 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Phone Service                      7043 non-null   object 
 10  Avg Monthly Long Distance Charges  7043 non-null   float64
 11  Multiple Lines                     7043 non-null   objec

In [0]:
data.columns = data.columns.str.replace(' ','_')

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 41 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer_ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number_of_Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip_Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Phone_Service                      7043 non-null   object 
 10  Avg_Monthly_Long_Distance_Charges  7043 non-null   float64
 11  Multiple_Lines                     7043 non-null   objec

In [0]:
data.head(3)

,Customer_ID,Gender,Age,Married,Number_of_Dependents,City,Zip_Code,Latitude,Longitude,Phone_Service,Avg_Monthly_Long_Distance_Charges,Multiple_Lines,Internet_Service,Avg_Monthly_GB_Download,Online_Security,Online_Backup,Device_Protection_Plan,Premium_Tech_Support,Streaming_TV,Streaming_Movies,Streaming_Music,Unlimited_Data,Internet_Type_Clean,Offer_Clean,Number_of_Referrals,Tenure_in_Months,Contract,Paperless_Billing,Payment_Method,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue,Customer_Status,Churn_Category_Clean,Churn_Reason_Clean,Has_Discount,Monthly_Discount_Amount,Population
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,Yes,42.39,No,Yes,16.0,No,Yes,No,Yes,Yes,No,No,Yes,Cable,No Offer,2,9,One Year,Yes,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,Not Churned,Not Churned,0,0.0,4498
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,Yes,10.69,Yes,Yes,10.0,No,No,No,No,No,Yes,Yes,No,Cable,No Offer,0,9,Month-to-Month,No,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,Not Churned,Not Churned,1,4.0,31297
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,Yes,33.65,No,Yes,30.0,No,No,Yes,No,No,No,No,Yes,Fiber Optic,Offer E,0,4,Month-to-Month,Yes,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices,0,0.0,62069


In [0]:
selected_columns = ['Age','Number_of_Dependents','Latitude','Longitude','Avg_Monthly_GB_Download','Avg_Monthly_Long_Distance_Charges','Number_of_Referrals','Tenure_in_Months','Monthly_Charge','Total_Charges','Total_Refunds','Total_Extra_Data_Charges','Total_Long_Distance_Charges','Total_Revenue','Monthly_Discount_Amount']

In [0]:
data['Gender'] = pd.get_dummies(data['Gender'], drop_first=True, dtype='int')
data['Married'] = np.where(data['Married'] == 'Yes', 1, 0)
data['Phone_Service'] = np.where(data['Phone_Service'] == 'Yes', 1, 0)
data['Multiple_Lines'] = np.where(data['Multiple_Lines'] == 'Yes', 1, 0)
data['Internet_Service'] = np.where(data['Internet_Service'] == 'Yes', 1, 0)
data['Online_Security'] = np.where(data['Online_Security'] == 'Yes', 1, 0)
data['Online_Backup'] = np.where(data['Online_Backup'] == 'Yes', 1, 0)
data['Device_Protection_Plan'] = np.where(data['Device_Protection_Plan'] == 'Yes', 1, 0)
data['Premium_Tech_Support'] = np.where(data['Premium_Tech_Support'] == 'Yes', 1, 0)
data['Streaming_TV'] = np.where(data['Streaming_TV'] == 'Yes', 1, 0)
data['Streaming_Movies'] = np.where(data['Streaming_Movies'] == 'Yes', 1, 0)
data['Streaming_Music'] = np.where(data['Streaming_Music'] == 'Yes', 1, 0)
data['Unlimited_Data'] = np.where(data['Unlimited_Data'] == 'Yes', 1, 0)
data = pd.concat([data, pd.get_dummies(data['Internet_Type_Clean'], drop_first=True,dtype='int', prefix='Internet_Type_Clean')], axis=1)
data.drop('Internet_Type_Clean', axis=1, inplace=True)
data = pd.concat([data, pd.get_dummies(data['Offer_Clean'], drop_first=True, dtype='int',prefix='Offer_Clean')], axis=1)
data.drop('Offer_Clean', axis=1, inplace=True)
data = pd.concat([data, pd.get_dummies(data['Contract'], drop_first=True,dtype='int', prefix='Contract')], axis=1)
data.drop('Contract', axis=1, inplace=True)
data['Paperless_Billing'] = np.where(data['Paperless_Billing'] == 'Yes', 1, 0)
data = pd.concat([data, pd.get_dummies(data['Payment_Method'], drop_first=True,dtype='int', prefix='Payment_Method')], axis=1)
data.drop('Payment_Method', axis=1, inplace=True)
data['Customer_Status'] = np.where(data['Customer_Status'] == 'Churned', 1, 0)
data = pd.concat([data, pd.get_dummies(data['Churn_Category_Clean'], drop_first=True, dtype='int',prefix='Churn_Category_Clean')], axis=1)
data.drop('Churn_Category_Clean', axis=1, inplace=True)
data = pd.concat([data, pd.get_dummies(data['Churn_Reason_Clean'], drop_first=True, dtype='int',prefix='Churn_Reason_Clean')], axis=1)
data.drop('Churn_Reason_Clean', axis=1, inplace=True)

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 72 columns):
 #   Column                                                        Non-Null Count  Dtype  
---  ------                                                        --------------  -----  
 0   Customer_ID                                                   7043 non-null   object 
 1   Gender                                                        7043 non-null   int64  
 2   Age                                                           7043 non-null   int64  
 3   Married                                                       7043 non-null   int64  
 4   Number_of_Dependents                                          7043 non-null   int64  
 5   City                                                          7043 non-null   object 
 6   Zip_Code                                                      7043 non-null   int64  
 7   Latitude                                                      7043 no

In [0]:
data.columns = data.columns.str.replace(' ','_')


In [0]:
selected_columns = ['Age','Number_of_Dependents','Latitude','Longitude','Avg_Monthly_GB_Download','Avg_Monthly_Long_Distance_Charges','Number_of_Referrals','Tenure_in_Months','Monthly_Charge','Total_Charges','Total_Refunds','Total_Extra_Data_Charges','Total_Long_Distance_Charges','Total_Revenue','Monthly_Discount_Amount']

In [0]:
data_new = data[selected_columns]

In [0]:
data_new.head()

,Age,Number_of_Dependents,Latitude,Longitude,Avg_Monthly_GB_Download,Avg_Monthly_Long_Distance_Charges,Number_of_Referrals,Tenure_in_Months,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue,Monthly_Discount_Amount
0,37,0,34.827662,-118.999073,16.0,42.39,2,9,65.6,593.30,0.00,0,381.51,974.81,0.0
1,46,0,34.162515,-118.203869,10.0,10.69,0,9,-4.0,542.40,38.33,10,96.21,610.28,4.0
2,50,0,33.645672,-117.922613,30.0,33.65,0,4,73.9,280.85,0.00,0,134.60,415.45,0.0
3,78,0,38.014457,-122.115432,4.0,27.82,1,13,98.0,1237.85,0.00,0,361.66,1599.51,0.0
4,75,0,34.227846,-119.079903,11.0,7.38,3,3,83.9,267.40,0.00,0,22.14,289.54,0.0


# Feature engineering

In [0]:
vif = pd.DataFrame()
vif['feature'] = data_new.columns
vif['score'] = [variance_inflation_factor(data_new.values, i) for i in range(len(data_new.columns))]

/local_disk0/.ephemeral_nfs/envs/pythonEnv-43ccbde9-e3a9-4d26-8a6f-d5b6cfeec564/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/local_disk0/.ephemeral_nfs/envs/pythonEnv-43ccbde9-e3a9-4d26-8a6f-d5b6cfeec564/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/local_disk0/.ephemeral_nfs/envs/pythonEnv-43ccbde9-e3a9-4d26-8a6f-d5b6cfeec564/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/local_disk0/.ephemeral_nfs/envs/pythonEnv-43ccbde9-e3a9-4d26-8a6f-d5b6cfeec564/lib/p

In [0]:
vif.loc[vif['score']<5, 'feature'].values

array(['Age', 'Number_of_Dependents', 'Latitude', 'Longitude',
       'Avg_Monthly_GB_Download', 'Avg_Monthly_Long_Distance_Charges',
       'Number_of_Referrals', 'Monthly_Charge', 'Monthly_Discount_Amount'],
      dtype=object)

In [0]:
segment_data = data[['Age', 'Number_of_Dependents', 'Latitude', 'Longitude',
       'Avg_Monthly_GB_Download', 'Avg_Monthly_Long_Distance_Charges',
       'Number_of_Referrals', 'Monthly_Charge', 'Monthly_Discount_Amount']]

# Standardization

In [0]:
std = StandardScaler()
X_scaled = std.fit_transform(segment_data)

# Elbow method + silhouette score to choose k

# K- Means clustering

In [0]:
inertia = []
silhouette = []

for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=123, n_init=10).fit(X_scaled)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(X_scaled, km.labels_))

In [0]:
results = list(zip(range(2,8), inertia, silhouette))

for k, i, s in sorted(results, key=lambda x: x[2], reverse=True):
    print(f'k={k}: inertia={i:.0f}, silhouette={s:.3f}')

k=5: inertia=35761, silhouette=0.216
k=6: inertia=33149, silhouette=0.216
k=4: inertia=40135, silhouette=0.207
k=3: inertia=45708, silhouette=0.203
k=2: inertia=52216, silhouette=0.194
k=7: inertia=31075, silhouette=0.190


# Fit final model

In [0]:
K = 5

In [0]:
kmeans = KMeans(n_clusters=K, random_state=123, n_init=10)

In [0]:
segment_data['Cluster'] = kmeans.fit_predict(X_scaled)

In [0]:
customer_cluster_data = pd.concat([data['Customer_ID'], segment_data['Cluster']], axis=1)
customer_cluster_data

,Customer_ID,Cluster
0,0002-ORFBO,2
1,0003-MKNFE,4
2,0004-TLHLJ,2
3,0011-IGKFF,1
4,0013-EXCHZ,2
...,...,...
7038,9987-LUTYD,0
7039,9992-RRAMN,1
7040,9992-UJOEL,0
7041,9993-LHIEB,0


# Profile each cluster

In [0]:
data.columns

Index(['Customer_ID', 'Gender', 'Age', 'Married', 'Number_of_Dependents',
       'City', 'Zip_Code', 'Latitude', 'Longitude', 'Phone_Service',
       'Avg_Monthly_Long_Distance_Charges', 'Multiple_Lines',
       'Internet_Service', 'Avg_Monthly_GB_Download', 'Online_Security',
       'Online_Backup', 'Device_Protection_Plan', 'Premium_Tech_Support',
       'Streaming_TV', 'Streaming_Movies', 'Streaming_Music', 'Unlimited_Data',
       'Number_of_Referrals', 'Tenure_in_Months', 'Paperless_Billing',
       'Monthly_Charge', 'Total_Charges', 'Total_Refunds',
       'Total_Extra_Data_Charges', 'Total_Long_Distance_Charges',
       'Total_Revenue', 'Customer_Status', 'Has_Discount',
       'Monthly_Discount_Amount', 'Population', 'Internet_Type_Clean_DSL',
       'Internet_Type_Clean_Fiber_Optic',
       'Internet_Type_Clean_No_Internet_Service', 'Offer_Clean_Offer_A',
       'Offer_Clean_Offer_B', 'Offer_Clean_Offer_C', 'Offer_Clean_Offer_D',
       'Offer_Clean_Offer_E', 'Contract_One_Yea

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 72 columns):
 #   Column                                                        Non-Null Count  Dtype  
---  ------                                                        --------------  -----  
 0   Customer_ID                                                   7043 non-null   object 
 1   Gender                                                        7043 non-null   int64  
 2   Age                                                           7043 non-null   int64  
 3   Married                                                       7043 non-null   int64  
 4   Number_of_Dependents                                          7043 non-null   int64  
 5   City                                                          7043 non-null   object 
 6   Zip_Code                                                      7043 non-null   int64  
 7   Latitude                                                      7043 no

In [0]:
data.columns

Index(['Customer_ID', 'Gender', 'Age', 'Married', 'Number_of_Dependents',
       'City', 'Zip_Code', 'Latitude', 'Longitude', 'Phone_Service',
       'Avg_Monthly_Long_Distance_Charges', 'Multiple_Lines',
       'Internet_Service', 'Avg_Monthly_GB_Download', 'Online_Security',
       'Online_Backup', 'Device_Protection_Plan', 'Premium_Tech_Support',
       'Streaming_TV', 'Streaming_Movies', 'Streaming_Music', 'Unlimited_Data',
       'Number_of_Referrals', 'Tenure_in_Months', 'Paperless_Billing',
       'Monthly_Charge', 'Total_Charges', 'Total_Refunds',
       'Total_Extra_Data_Charges', 'Total_Long_Distance_Charges',
       'Total_Revenue', 'Customer_Status', 'Has_Discount',
       'Monthly_Discount_Amount', 'Population', 'Internet_Type_Clean_DSL',
       'Internet_Type_Clean_Fiber_Optic',
       'Internet_Type_Clean_No_Internet_Service', 'Offer_Clean_Offer_A',
       'Offer_Clean_Offer_B', 'Offer_Clean_Offer_C', 'Offer_Clean_Offer_D',
       'Offer_Clean_Offer_E', 'Contract_One_Yea

In [0]:
X = data.drop(['Customer_ID','City','Zip_Code'], axis=1)

In [0]:
X = data.drop(['Customer_ID','City','Zip_Code'], axis=1)
X['Cluster'] = segment_data['Cluster']

In [0]:
cluster_profile = X.groupby('Cluster').mean()
cluster_profile['Customer_Count'] = X.groupby('Cluster').size()
cluster_profile.columns

Index(['Gender', 'Age', 'Married', 'Number_of_Dependents', 'Latitude',
       'Longitude', 'Phone_Service', 'Avg_Monthly_Long_Distance_Charges',
       'Multiple_Lines', 'Internet_Service', 'Avg_Monthly_GB_Download',
       'Online_Security', 'Online_Backup', 'Device_Protection_Plan',
       'Premium_Tech_Support', 'Streaming_TV', 'Streaming_Movies',
       'Streaming_Music', 'Unlimited_Data', 'Number_of_Referrals',
       'Tenure_in_Months', 'Paperless_Billing', 'Monthly_Charge',
       'Total_Charges', 'Total_Refunds', 'Total_Extra_Data_Charges',
       'Total_Long_Distance_Charges', 'Total_Revenue', 'Customer_Status',
       'Has_Discount', 'Monthly_Discount_Amount', 'Population',
       'Internet_Type_Clean_DSL', 'Internet_Type_Clean_Fiber_Optic',
       'Internet_Type_Clean_No_Internet_Service', 'Offer_Clean_Offer_A',
       'Offer_Clean_Offer_B', 'Offer_Clean_Offer_C', 'Offer_Clean_Offer_D',
       'Offer_Clean_Offer_E', 'Contract_One_Year', 'Contract_Two_Year',
       'Payment_M

# Saving the model to pkl file and to csv file

In [0]:
joblib.dump(kmeans, '/Volumes/workspace/default/raw_data/segmentation_model.pkl')
cluster_profile.to_csv('/Volumes/workspace/default/raw_data/cluster_profile.csv', index=False)
customer_cluster_data.to_csv('/Volumes/workspace/default/raw_data/customer_cluster_data.csv', index=False)

In [0]:
joblib.load('/Volumes/workspace/default/raw_data/segmentation_model.pkl')

KMeans(n_clusters=5, n_init=10, random_state=123)

In [0]:
pd.read_csv('/Volumes/workspace/default/raw_data/cluster_profile.csv')

,Gender,Age,Married,Number_of_Dependents,Latitude,Longitude,Phone_Service,Avg_Monthly_Long_Distance_Charges,Multiple_Lines,Internet_Service,Avg_Monthly_GB_Download,Online_Security,Online_Backup,Device_Protection_Plan,Premium_Tech_Support,Streaming_TV,Streaming_Movies,Streaming_Music,Unlimited_Data,Number_of_Referrals,Tenure_in_Months,Paperless_Billing,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue,Customer_Status,Has_Discount,Monthly_Discount_Amount,Population,Internet_Type_Clean_DSL,Internet_Type_Clean_Fiber_Optic,Internet_Type_Clean_No_Internet_Service,Offer_Clean_Offer_A,Offer_Clean_Offer_B,Offer_Clean_Offer_C,Offer_Clean_Offer_D,Offer_Clean_Offer_E,Contract_One_Year,Contract_Two_Year,Payment_Method_Credit_Card,Payment_Method_Mailed_Check,Churn_Category_Clean_Competitor,Churn_Category_Clean_Dissatisfaction,Churn_Category_Clean_Not_Churned,Churn_Category_Clean_Other,Churn_Category_Clean_Price,Churn_Reason_Clean_Attitude_of_support_person,Churn_Reason_Clean_Competitor_had_better_devices,Churn_Reason_Clean_Competitor_made_better_offer,Churn_Reason_Clean_Competitor_offered_higher_download_speeds,Churn_Reason_Clean_Competitor_offered_more_data,Churn_Reason_Clean_Deceased,Churn_Reason_Clean_Don't_know,Churn_Reason_Clean_Extra_data_charges,Churn_Reason_Clean_Lack_of_affordable_download/upload_speed,Churn_Reason_Clean_Lack_of_self-service_on_Website,Churn_Reason_Clean_Limited_range_of_services,Churn_Reason_Clean_Long_distance_charges,Churn_Reason_Clean_Moved,Churn_Reason_Clean_Network_reliability,Churn_Reason_Clean_Not_Churned,Churn_Reason_Clean_Poor_expertise_of_online_support,Churn_Reason_Clean_Poor_expertise_of_phone_support,Churn_Reason_Clean_Price_too_high,Churn_Reason_Clean_Product_dissatisfaction,Churn_Reason_Clean_Service_dissatisfaction,Customer_Count
0,0.486065,24.286511,0.437012,0.176143,35.870329,-119.493218,0.880713,26.068055,0.458194,0.998885,58.937781,0.401338,0.445931,0.442586,0.402453,0.491639,0.516165,0.639911,0.850613,1.646600,32.364548,0.663322,76.092642,2703.046600,1.658852,9.442586,741.006187,3451.836522,0.302118,0.004459,0.008919,24789.919732,0.309922,0.519509,0.001115,0.091416,0.118172,0.070234,0.069119,0.130435,0.221851,0.201784,0.352285,0.057971,0.146042,0.057971,0.697882,0.028986,0.021182,0.027871,0.046823,0.051282,0.025641,0.022297,0.001115,0.023411,0.002230,0.003344,0.006689,0.007804,0.005574,0.004459,0.017837,0.697882,0.004459,0.002230,0.010033,0.012263,0.006689,897
1,0.510408,50.954363,0.404724,0.115693,38.482097,-121.710236,0.903923,25.679297,0.421938,0.784227,18.873654,0.279023,0.328263,0.339071,0.279023,0.378703,0.393915,0.328663,0.893515,1.509207,31.259808,0.602882,64.659287,2211.649720,1.879031,6.313050,733.667058,2949.750797,0.295436,0.004804,0.008407,14519.839872,0.227382,0.446357,0.215773,0.061249,0.112490,0.050841,0.092474,0.114492,0.208167,0.253803,0.371898,0.058847,0.126101,0.056845,0.704564,0.030024,0.040032,0.029223,0.058046,0.025220,0.018415,0.024420,0.001201,0.019616,0.007606,0.004804,0.004404,0.006405,0.011609,0.009207,0.010809,0.704564,0.006405,0.002802,0.016013,0.014011,0.012010,2498
2,0.500193,51.759552,0.432266,0.139714,33.996473,-117.864910,0.911231,25.026680,0.431880,0.788885,18.646047,0.252412,0.342339,0.337321,0.269008,0.389811,0.383636,0.307989,0.888460,1.549981,31.294481,0.616364,65.255191,2253.291625,2.076333,7.005017,722.060583,2980.280892,0.308761,0.005789,0.011579,29040.930529,0.218063,0.461984,0.211115,0.074489,0.114242,0.061366,0.081436,0.126206,0.227711,0.243921,0.372057,0.052875,0.145118,0.043998,0.691239,0.030104,0.033578,0.041297,0.045928,0.074103,0.011193,0.013894,0.000772,0.022385,0.006561,0.005789,0.003860,0.005403,0.010421,0.006947,0.010035,0.691239,0.004245,0.001158,0.010807,0.011193,0.008105,2591
3,0.513875,41.474820,0.864337,2.514902,36.516708,-120.031430,0.900308,25.340709,0.362795,0.568345,34.798338,0.291881,0.302158,0.288798,0.267215,0.289825,0.270298,0.273381,0.926002,4.435766,38.434738,0.4409

In [0]:
pd.read_csv('/Volumes/workspace/default/raw_data/customer_cluster_data.csv')

,Customer_ID,Cluster
0,0002-ORFBO,2
1,0003-MKNFE,4
2,0004-TLHLJ,2
3,0011-IGKFF,1
4,0013-EXCHZ,2
...,...,...
7038,9987-LUTYD,0
7039,9992-RRAMN,1
7040,9992-UJOEL,0
7041,9993-LHIEB,0


#End